# Notebook 1 — Foundations: the occupancy library, validated
**Goal.** Build `occlib` (exact occupancy statistics, high-pass ratio, surrogates, pool-growth score, simulators) and check it three ways before any real data touches it:
1. **Unit tests** against brute-force time-grid computations.
2. **Theory checks**: Proposition 2 (hidden pool), Proposition 3 (slot-count modulation), open-loop relays, pinned workers.
3. **Calibration and power** on simulated benign pools: null spread vs window, correlation time κ, power vs attacker jitter, simple baselines, hidden-worker estimate, pool-growth AUC.

**Run settings.** Accelerator: *None* (CPU is enough; save your GPU quota). Internet: *On* (only used by the environment probe). Expected runtime: 15–30 min.

**What to paste back.** Only the block printed by the last cell, between `===== PASTE FROM HERE =====` and `===== PASTE TO HERE =====`.

In [ ]:
# Cell 1 — environment probe (information for planning later notebooks)
import sys, os, platform, subprocess, shutil, json, time
env = {"python": sys.version.split()[0], "platform": platform.platform()}
try:
    import numpy, scipy, sklearn
    env.update(numpy=numpy.__version__, scipy=scipy.__version__, sklearn=sklearn.__version__)
except Exception as ex:
    env["import_error"] = repr(ex)
try:
    import torch
    env["torch"] = torch.__version__
    env["cuda_available"] = torch.cuda.is_available()
    if torch.cuda.is_available():
        env["gpus"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
        env["gpu_capability"] = [".".join(map(str, torch.cuda.get_device_capability(i))) for i in range(torch.cuda.device_count())]
except Exception as ex:
    env["torch"] = f"not importable: {ex!r}"
env["cpu_count"] = os.cpu_count()
try:
    env["ram_gb"] = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1)
except Exception:
    pass
env["disk_free_gb_working"] = round(shutil.disk_usage("/kaggle/working" if os.path.exists("/kaggle/working") else ".").free / 1e9, 1)
try:
    import urllib.request
    urllib.request.urlopen("https://huggingface.co", timeout=10)
    env["internet"] = True
except Exception as ex:
    env["internet"] = f"no ({type(ex).__name__})"
try:
    from kaggle_secrets import UserSecretsClient
    try:
        UserSecretsClient().get_secret("HF_TOKEN"); env["hf_token_secret"] = True
    except Exception:
        env["hf_token_secret"] = False
except Exception:
    env["hf_token_secret"] = "not on kaggle"
print(json.dumps(env, indent=1))
T_START = time.time()


In [ ]:
# Cell 1b — data-access probe for Notebook 2 (WildChat: real human request timing). Downloads ~one shard at most.
probe = {}
try:
    from huggingface_hub import HfApi, hf_hub_download
    tok = None
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    api = HfApi()
    for repo in ["allenai/WildChat-1M", "allenai/WildChat-4.8M", "allenai/WildChat"]:
        try:
            files = [f for f in api.list_repo_tree(repo, repo_type="dataset", recursive=True, token=tok)
                     if getattr(f, "path", "").endswith(".parquet")]
            probe[repo] = dict(n_parquet=len(files), total_gb=round(sum((f.size or 0) for f in files) / 1e9, 2),
                               first=files[0].path if files else None)
        except Exception as ex:
            probe[repo] = f"error: {type(ex).__name__}: {str(ex)[:120]}"
    # try one shard of WildChat-1M and report its schema
    r = probe.get("allenai/WildChat-1M")
    if isinstance(r, dict) and r["first"]:
        import pyarrow.parquet as pq
        p = hf_hub_download("allenai/WildChat-1M", r["first"], repo_type="dataset", token=tok)
        pf = pq.ParquetFile(p)
        probe["wildchat1m_shard_rows"] = pf.metadata.num_rows
        probe["wildchat1m_schema"] = str(pf.schema_arrow)[:1500]
except Exception as ex:
    probe["error"] = f"{type(ex).__name__}: {str(ex)[:200]}"
print(json.dumps(probe, indent=1)[:4000])
env["data_probe"] = {k: (v if k != "wildchat1m_schema" else v[:800]) for k, v in probe.items()}


In [ ]:
%%writefile occlib.py
"""
occlib — occupancy statistics for detecting hidden closed-loop worker pools behind many accounts.

Data model: a request log as three equal-length arrays
    acct : int   account index 0..A-1
    s    : float request arrival time (seconds)
    e    : float time the streamed response finished (seconds), e >= s
X_i(t) = number of account i's requests in flight; N(t) = sum_i X_i(t).

All statistics are exact time integrals of step functions (no time grid).
"""
import numpy as np

__version__ = "0.1.0"

# --------------------------------------------------------------------------------------------
# core: exact per-group integrals of X and X^2
# --------------------------------------------------------------------------------------------
def _group_integrals(gid, s, e, ngroups):
    """For intervals carrying a group id, return I1[g] = int X_g dt and I2[g] = int X_g^2 dt,
    where X_g(t) = number of the group's intervals covering t (half-open [s, e))."""
    I1 = np.zeros(ngroups); I2 = np.zeros(ngroups)
    n = len(s)
    if n == 0:
        return I1, I2
    t = np.concatenate([s, e])
    d = np.concatenate([np.ones(n), -np.ones(n)])
    g = np.concatenate([gid, gid])
    o = np.lexsort((d, t, g))          # by group, then time, then ends before starts
    t, d, g = t[o], d[o], g[o]
    x = np.cumsum(d)                    # each group nets to zero, so the level resets between groups
    dt = np.empty_like(t)
    dt[:-1] = t[1:] - t[:-1]
    dt[-1] = 0.0
    dt[:-1][g[1:] != g[:-1]] = 0.0
    I1 += np.bincount(g, weights=x * dt, minlength=ngroups)
    I2 += np.bincount(g, weights=x * x * dt, minlength=ngroups)
    return I1, I2


def _clip(acct, s, e, t0, t1):
    s = np.clip(np.asarray(s, float), t0, t1); e = np.clip(np.asarray(e, float), t0, t1)
    m = e > s
    return np.asarray(acct)[m], s[m], e[m]


def _split_blocks(acct, s, e, t0, tau, nb):
    """Split intervals at block boundaries t0 + k*tau. Returns (acct, block, s, e) pieces."""
    b0 = np.floor((s - t0) / tau).astype(np.int64)
    b1 = np.floor((np.nextafter(e, -np.inf) - t0) / tau).astype(np.int64)
    b0 = np.clip(b0, 0, nb - 1); b1 = np.clip(b1, 0, nb - 1)
    cnt = b1 - b0 + 1
    idx = np.repeat(np.arange(len(s)), cnt)
    off = np.arange(cnt.sum()) - np.repeat(np.cumsum(cnt) - cnt, cnt)
    blk = b0[idx] + off
    ps = np.maximum(s[idx], t0 + blk * tau)
    pe = np.minimum(e[idx], t0 + (blk + 1) * tau)
    m = pe > ps
    return acct[idx][m], blk[m], ps[m], pe[m]


def occupancy(acct, s, e, t0, t1, tau=None, n_accounts=None):
    """Occupancy moments over [t0, t1].
    tau=None -> full-window variances; tau=float -> mean over blocks of within-block variances
    (the 'high-pass' statistic). Returns dict with R, V_N, V_i (per account), mean_N, n_blocks."""
    acct = np.asarray(acct, np.int64)
    A = int(n_accounts if n_accounts is not None else (acct.max() + 1 if len(acct) else 0))
    if tau is None:
        tau = t1 - t0
    nb = int(np.floor((t1 - t0) / tau + 1e-9))
    if nb < 1:
        raise ValueError("window shorter than one block")
    t1 = t0 + nb * tau
    a, s, e = _clip(acct, s, e, t0, t1)
    a, b, s, e = _split_blocks(a, s, e, t0, tau, nb)
    # aggregate N, per block
    I1, I2 = _group_integrals(b, s, e, nb)
    VN_b = I2 / tau - (I1 / tau) ** 2
    # per account per block (compress group ids)
    key = a * nb + b
    uk, inv = np.unique(key, return_inverse=True)
    J1, J2 = _group_integrals(inv, s, e, len(uk))
    Vg = J2 / tau - (J1 / tau) ** 2
    V_i = np.bincount((uk // nb).astype(np.int64), weights=Vg, minlength=A) / nb
    V_N = VN_b.mean()
    sV = V_i.sum()
    return dict(R=V_N / sV if sV > 0 else np.nan, V_N=V_N, V_i=V_i, sumV=sV,
                mean_N=I1.sum() / (nb * tau), n_blocks=nb, tau=tau)


def R_full(acct, s, e, t0, t1):
    return occupancy(acct, s, e, t0, t1, None)["R"]


def R_hp(acct, s, e, t0, t1, tau=60.0):
    return occupancy(acct, s, e, t0, t1, tau)["R"]


def fano_khat(acct, s, e, t0, t1):
    """Fano factor of N and the implied effective number of hidden slots K = <N>^2/(<N>-Var N)."""
    o = occupancy(acct, s, e, t0, t1, None)
    F = o["V_N"] / o["mean_N"] if o["mean_N"] > 0 else np.nan
    p = 1 - F
    K = o["mean_N"] / p if p > 0 else np.inf
    return dict(F=F, p_hat=p, K_hat=K, mean_N=o["mean_N"], V_N=o["V_N"])


def baselines(acct, s, e, t0, t1):
    """Simple baselines: requests/hour, peak concurrency, Fano factor of N."""
    a, sc, ec = _clip(acct, s, e, t0, t1)
    n_req = int(((np.asarray(s) >= t0) & (np.asarray(s) < t1)).sum())
    t = np.concatenate([sc, ec]); d = np.concatenate([np.ones(len(sc)), -np.ones(len(ec))])
    o = np.lexsort((d, t)); peak = int(np.cumsum(d[o]).max()) if len(t) else 0
    f = fano_khat(acct, s, e, t0, t1)
    return dict(req_per_h=n_req / ((t1 - t0) / 3600), peak_N=peak, fano=f["F"])


# --------------------------------------------------------------------------------------------
# nulls
# --------------------------------------------------------------------------------------------
def circular_shift(acct, s, e, t0, t1, rng):
    """Shift each account's timeline by an independent uniform offset, wrapping around [t0, t1).
    Keeps each account's marginal behaviour and autocorrelation; destroys cross-account dependence."""
    acct = np.asarray(acct, np.int64); L = t1 - t0
    a, s, e = _clip(acct, s, e, t0, t1)
    A = int(acct.max() + 1) if len(acct) else 0
    off = rng.uniform(0, L, A)[a]
    s2 = t0 + np.mod(s - t0 + off, L); e2 = s2 + (e - s)
    wrap = e2 > t1
    a_out = np.concatenate([a, a[wrap]])
    s_out = np.concatenate([s2, np.full(wrap.sum(), t0)])
    e_out = np.concatenate([np.minimum(e2, t1), t0 + (e2[wrap] - t1)])
    return a_out, s_out, e_out


def surrogate_null(acct, s, e, t0, t1, tau, n_sur, rng):
    """Distribution of R (tau=None) or R_tau under independent circular shifts."""
    return np.array([occupancy(*circular_shift(acct, s, e, t0, t1, rng), t0, t1, tau)["R"] for _ in range(n_sur)])


# --------------------------------------------------------------------------------------------
# pool growth: correlation of each candidate's occupancy with a pool's aggregate occupancy
# --------------------------------------------------------------------------------------------
def _cum_integral(s, e, t0, t1):
    """Knots and running integral F(t) = int_t0^t N(u) du for the superposition of intervals."""
    t = np.concatenate([[t0], s, e, [t1]])
    d = np.concatenate([[0.0], np.ones(len(s)), -np.ones(len(e)), [0.0]])
    o = np.lexsort((d, t)); t, d = t[o], d[o]
    x = np.cumsum(d)
    F = np.concatenate([[0.0], np.cumsum(x[:-1] * np.diff(t))])
    return t, F


def corr_with_pool(pool, cand, t0, t1, tau=None, n_cand=None):
    """pool=(acct,s,e) of pool members; cand=(acct,s,e) of candidates (acct = candidate index 0..C-1).
    Returns per-candidate correlation of X_j with N_pool (within-block if tau is set). Linear in traffic."""
    pa, ps, pe = _clip(*pool, t0, t1)
    ca, cs, ce = _clip(*cand, t0, t1)
    C = int(n_cand if n_cand is not None else np.asarray(cand[0]).max() + 1)
    if tau is None:
        tau = t1 - t0
    nb = int(np.floor((t1 - t0) / tau + 1e-9)); t1 = t0 + nb * tau
    # pool per-block mean and within-block variance
    _, pb, pps, ppe = _split_blocks(pa, np.clip(ps, t0, t1), np.clip(pe, t0, t1), t0, tau, nb)
    I1, I2 = _group_integrals(pb, pps, ppe, nb)
    Nbar = I1 / tau; VN = (I2 / tau - Nbar ** 2).mean()
    kn, F = _cum_integral(np.clip(ps, t0, t1), np.clip(pe, t0, t1), t0, t1)
    # candidate pieces
    ca2, cb, cs2, ce2 = _split_blocks(ca, np.clip(cs, t0, t1), np.clip(ce, t0, t1), t0, tau, nb)
    XN = np.interp(ce2, kn, F) - np.interp(cs2, kn, F)       # int over piece of N_pool
    key = ca2 * nb + cb
    uk, inv = np.unique(key, return_inverse=True)
    A = np.bincount(inv, weights=XN)                          # int X_j N over block
    B = np.bincount(inv, weights=ce2 - cs2)                   # int X_j over block
    jj = uk // nb; bb = uk % nb
    cov = np.bincount(jj, weights=A / tau - (B / tau) * Nbar[bb], minlength=C) / nb
    Vj = occupancy(ca, cs, ce, t0, t1, None if nb == 1 else tau, n_accounts=C)["V_i"]
    with np.errstate(invalid="ignore", divide="ignore"):
        return cov / np.sqrt(Vj * VN)


# --------------------------------------------------------------------------------------------
# generators (simulation)
# --------------------------------------------------------------------------------------------
DUR_MED, DUR_SIG = 8.0, 0.6       # streamed response duration, log-normal
GAP_MED, GAP_SIG = 0.4, 0.5       # client processing gap, log-normal
E_DUR = DUR_MED * np.exp(DUR_SIG ** 2 / 2)
E_GAP = GAP_MED * np.exp(GAP_SIG ** 2 / 2)


def duty(J):
    return E_DUR / (E_DUR + E_GAP + J / 2.0)


def theory_R(p, M):
    return (1 - p) / (1 - p / M)


def _pack(rows):
    if not rows:
        return np.zeros(0, np.int64), np.zeros(0), np.zeros(0)
    a, s, e = map(np.asarray, zip(*rows))
    return a.astype(np.int64), s.astype(float), e.astype(float)


def benign_humans(rng, n_accounts, T, session_rate=1 / 7200, mean_req=6, think_med=25.0, think_sig=0.9,
                  diurnal=0.0, offset=0):
    """Human-like accounts: Poisson session starts, geometric #requests, log-normal think time.
    diurnal in [0,1): amplitude of a 24 h sinusoidal rate modulation (common to all accounts)."""
    rows = []
    for i in range(n_accounts):
        t = rng.exponential(1 / session_rate)
        while t < T:
            if diurnal > 0 and rng.random() > (1 + diurnal * np.sin(2 * np.pi * t / 86400)) / (1 + diurnal):
                t += rng.exponential(1 / session_rate); continue
            tt = t
            for _ in range(rng.geometric(1 / mean_req)):
                if tt > T: break
                d = rng.lognormal(np.log(DUR_MED), DUR_SIG); rows.append((offset + i, tt, tt + d))
                tt += d + rng.lognormal(np.log(think_med), think_sig)
            t = tt + rng.exponential(1 / session_rate)
    return _pack(rows)


def closed_loop_pool(rng, T, K, M, J=0.0, assign="random", stint=None, K_bar=None, epoch=None, offset=0):
    """K closed-loop slots over M accounts. Each slot: send, wait for stream end, gap, uniform(0,J) delay.
    assign: 'random' (per request), 'pinned' (slot w -> account w % M), 'stint' (hold an account for `stint` s).
    K_bar/epoch: if set, the number of active slots is redrawn ~ Poisson(K_bar) (capped at K) every `epoch` s."""
    rows = []
    Kact = None
    if K_bar is not None:
        Kact = np.minimum(rng.poisson(K_bar, int(T // epoch) + 2), K)
    held = {}
    for w in range(K):
        t = rng.uniform(0, 5)
        while t < T:
            if Kact is not None:
                ei = int(t // epoch)
                if w >= Kact[ei]:
                    t = (ei + 1) * epoch + rng.uniform(0, 1); continue
            if assign == "pinned":
                a = w % M
            elif assign == "stint":
                k = (w, int(t // stint))
                if k not in held: held[k] = rng.integers(M)
                a = held[k]
            else:
                a = rng.integers(M)
            d = rng.lognormal(np.log(DUR_MED), DUR_SIG)
            rows.append((offset + a, t, t + d))
            t += d + rng.lognormal(np.log(GAP_MED), GAP_SIG) + (rng.uniform(0, J) if J > 0 else 0.0)
    return _pack(rows)


def open_loop_relay(rng, T, M, lam, policy="random", offset=0):
    """Open-loop Poisson arrivals (rate lam/s) dispatched over M accounts: random | rr | least."""
    import heapq
    rows = []; inflight = np.zeros(M, int); heap = []; t = 0.0; rr = 0
    while True:
        t += rng.exponential(1 / lam)
        if t > T: break
        while heap and heap[0][0] <= t:
            _, a = heapq.heappop(heap); inflight[a] -= 1
        if policy == "random": a = int(rng.integers(M))
        elif policy == "rr": a = rr % M; rr += 1
        else:
            m = inflight.min(); a = int(rng.choice(np.flatnonzero(inflight == m)))
        d = rng.lognormal(np.log(DUR_MED), DUR_SIG)
        rows.append((offset + a, t, t + d)); inflight[a] += 1; heapq.heappush(heap, (t + d, a))
    return _pack(rows)


def concat(*logs):
    return tuple(np.concatenate(x) for x in zip(*logs))


def relabel(acct):
    """Map arbitrary account ids to 0..A-1."""
    u, inv = np.unique(acct, return_inverse=True)
    return inv.astype(np.int64), u


In [ ]:
# Cell 2b — import the library
import importlib, occlib
importlib.reload(occlib)
print("occlib", occlib.__version__, "written to", os.path.abspath("occlib.py"))


In [ ]:
# Cell 3 — unit tests against brute force on a fine time grid
import numpy as np
from occlib import *
RES = {"env": env}
rng = np.random.default_rng(0)

def brute(acct, s, e, t0, t1, tau=None, dt=0.002):
    tau = (t1 - t0) if tau is None else tau
    nb = int(np.floor((t1 - t0) / tau + 1e-9)); t1 = t0 + nb * tau
    grid = np.arange(t0, t1, dt) + dt / 2
    A = acct.max() + 1
    X = np.zeros((A, len(grid)))
    for a, ss, ee in zip(acct, s, e):
        X[a] += (grid >= ss) & (grid < ee)
    N = X.sum(0)
    blk = ((grid - t0) // tau).astype(int)
    def wv(y):
        return np.mean([y[blk == b].var() for b in range(nb)])
    VN = wv(N); Vi = np.array([wv(X[a]) for a in range(A)])
    return VN / Vi.sum(), VN, Vi

tests = {}
# random small log with overlapping requests inside accounts
A = 7; n = 120
acct = rng.integers(0, A, n); s = rng.uniform(0, 300, n); e = s + rng.lognormal(1.5, 0.8, n)
for tau in [None, 37.0, 60.0]:
    ex = occupancy(acct, s, e, 10.0, 290.0, tau)
    rb, vnb, vib = brute(acct, s, e, 10.0, 290.0, tau)
    tests[f"exact_vs_grid_tau={tau}"] = float(max(abs(ex["R"] - rb), np.max(np.abs(ex["V_i"] - vib)), abs(ex["V_N"] - vnb)))
# identity: tau = whole window equals full-window
tests["Rtau_equals_R_when_tau_is_window"] = float(abs(R_hp(acct, s, e, 10, 290, 280.0) - R_full(acct, s, e, 10, 290)))
# surrogate keeps per-account variances
a2, s2, e2 = circular_shift(acct, s, e, 10, 290, rng)
tests["surrogate_keeps_V_i"] = float(np.max(np.abs(occupancy(a2, s2, e2, 10, 290)["V_i"] - occupancy(acct, s, e, 10, 290)["V_i"])))
# pool correlation against brute force
pool = (acct[acct < 4], s[acct < 4], e[acct < 4])
cm = acct >= 4
cand = (acct[cm] - 4, s[cm], e[cm])
c_ex = corr_with_pool(pool, cand, 10.0, 290.0)
grid = np.arange(10, 290, 0.002) + 0.001
Np = sum(((grid >= a) & (grid < b)).astype(float) for a, b in zip(*pool[1:]))
c_br = []
for j in range(3):
    m = cand[0] == j
    Xj = sum(((grid >= a) & (grid < b)).astype(float) for a, b in zip(cand[1][m], cand[2][m]))
    c_br.append(np.corrcoef(Xj, Np)[0, 1])
tests["corr_with_pool_vs_grid"] = float(np.max(np.abs(c_ex - np.array(c_br))))
RES["unit_tests_max_abs_error"] = tests
print(json.dumps(tests, indent=1))
assert all(v < 5e-3 for v in tests.values()), "unit test failed"
print("all unit tests passed")


In [ ]:
# Cell 4 — theory checks (Proposition 2, open loop, pinning, Proposition 3 modulation)
T0, T6 = 300.0, 6 * 3600.0
rows = []
for K in [2, 4, 8, 16, 34]:
    for J in [0, 10, 30, 60, 120]:
        for M in [10, 40]:
            Rs = [R_full(*closed_loop_pool(rng, T6, K, M, J), T0, T6) for _ in range(4)]
            rows.append(dict(K=K, J=J, M=M, R_sim=float(np.mean(Rs)), R_th=float(theory_R(duty(J), M))))
err = [abs(r["R_sim"] - r["R_th"]) for r in rows]
RES["prop2"] = dict(n_settings=len(rows), max_abs_err=round(max(err), 4), mean_abs_err=round(float(np.mean(err)), 4))
print("Prop 2:", RES["prop2"])

edge = {}
edge["pinned_40_slots_40_accts_R"] = float(np.mean([R_full(*closed_loop_pool(rng, T6, 40, 40, 0, assign="pinned"), T0, T6) for _ in range(2)]))
for pol in ["random", "rr", "least"]:
    edge[f"open_loop_relay_{pol}_R"] = float(np.mean([R_full(*open_loop_relay(rng, T6, 40, 0.2, pol), T0, T6) for _ in range(2)]))
RES["edge_cases"] = {k: round(v, 3) for k, v in edge.items()}
print("edge cases:", RES["edge_cases"])

mod = []
for label, kw in [("none", dict(K=10)), ("poisson10_every300s", dict(K=40, K_bar=10, epoch=300)),
                  ("poisson10_every60s", dict(K=40, K_bar=10, epoch=60)), ("poisson10_every20s", dict(K=40, K_bar=10, epoch=20)),
                  ("every300s_plus_30min_stints", dict(K=40, K_bar=10, epoch=300, assign="stint", stint=1800))]:
    v = []
    for _ in range(4):
        log = closed_loop_pool(rng, T6, M=40, **kw)
        v.append([R_full(*log, T0, T6), R_hp(*log, T0, T6, 60.0), R_hp(*log, T0, T6, 20.0)])
    v = np.mean(v, 0)
    mod.append(dict(case=label, R=round(float(v[0]), 3), R60=round(float(v[1]), 3), R20=round(float(v[2]), 3)))
RES["modulation"] = mod
print("Prop 3 / modulation:"); [print(" ", m) for m in mod]


In [ ]:
# Cell 5 — null calibration on simulated benign pools (40 accounts)
null = {}
for hours, npools in [(1, 400), (6, 400), (24, 120)]:
    T = hours * 3600.0
    R1, R2 = [], []
    for _ in range(npools):
        log = benign_humans(rng, 40, T)
        R1.append(R_full(*log, T0, T)); R2.append(R_hp(*log, T0, T, 60.0))
    R1, R2 = np.array(R1), np.array(R2)
    null[f"{hours}h"] = dict(n=npools, R_mean=round(float(R1.mean()), 4), R_sd=round(float(R1.std()), 4),
                             R60_mean=round(float(R2.mean()), 4), R60_sd=round(float(R2.std()), 4),
                             R60_q01=round(float(np.quantile(R2, 0.01)), 4))
L = np.array([h * 3600 - T0 for h in (1, 6, 24)])
null["kappa_R_s"] = round(float(np.mean([null[f"{h}h"]["R_sd"] ** 2 * l / 2 for h, l in zip((1, 6, 24), L)])), 2)
null["kappa_R60_s"] = round(float(np.mean([null[f"{h}h"]["R60_sd"] ** 2 * l / 2 for h, l in zip((1, 6, 24), L)])), 2)
# surrogate null vs reference null on single pools (6 h)
sur = []
for _ in range(5):
    log = benign_humans(rng, 40, T6)
    sur.append(float(surrogate_null(*log, T0, T6, 60.0, 40, rng).std()))
null["surrogate_sd_R60_6h_mean"] = round(float(np.mean(sur)), 4)
# common-mode daily cycle
Rd = [R_full(*benign_humans(rng, 40, 86400.0, diurnal=0.8), T0, 86400.0) for _ in range(20)]
null["diurnal_24h_R_mean"] = round(float(np.mean(Rd)), 3)
RES["null"] = null
print(json.dumps(null, indent=1))


In [ ]:
# Cell 6 — power vs attacker jitter at fixed throughput, baselines, hidden-worker estimate, pool growth
from sklearn.metrics import roc_auc_score
power = []
for hours in [1, 6]:
    T = hours * 3600.0; nn = null[f"{hours}h"]
    for J, K in [(0, 4), (10, 6), (30, 10), (60, 16), (120, 28), (240, 52)]:
        v = []; thr = []
        for _ in range(6):
            log = closed_loop_pool(rng, T, K, 40, J)
            v.append(R_hp(*log, T0, T, 60.0)); thr.append(baselines(*log, T0, T)["req_per_h"])
        power.append(dict(h=hours, J=J, K=K, req_h=int(np.mean(thr)), R60=round(float(np.mean(v)), 3),
                          z=round(float((nn["R60_mean"] - np.mean(v)) / nn["R60_sd"]), 1)))
RES["power"] = power
print("power:"); [print(" ", p) for p in power]

def heavy_benign():
    return concat(benign_humans(rng, 36, T6), closed_loop_pool(rng, T6, 4, 4, 0, assign="pinned", offset=36))
cases = [("benign_40_humans", lambda: benign_humans(rng, 40, T6)),
         ("benign_36_humans_plus_4_batch_clients", heavy_benign),
         ("attack_4_slots_40_accts", lambda: closed_loop_pool(rng, T6, 4, 40, 0)),
         ("attack_10_slots_J30", lambda: closed_loop_pool(rng, T6, 10, 40, 30)),
         ("legit_company_4_slots_40_keys", lambda: closed_loop_pool(rng, T6, 4, 40, 0))]
base = []
for name, gen in cases:
    log = gen(); b = baselines(*log, T0, T6)
    base.append(dict(pool=name, req_h=int(b["req_per_h"]), peak=int(b["peak_N"]), fano=round(float(b["fano"]), 3),
                     R=round(float(R_full(*log, T0, T6)), 3), R60=round(float(R_hp(*log, T0, T6, 60.0)), 3)))
RES["baselines"] = base
print("baselines:"); [print(" ", b) for b in base]

kh = []
for J, K in [(0, 4), (10, 6), (30, 10), (60, 18)]:
    kk = [fano_khat(*closed_loop_pool(rng, T6, K, 40, J), T0, T6)["K_hat"] for _ in range(3)]
    kh.append(dict(K=K, J=J, K_hat=round(float(np.mean(kk)), 2)))
RES["K_hat"] = kh
print("K_hat:", kh)

growth = []
for J, K in [(0, 4), (30, 10), (120, 28)]:
    aucs = []
    for _ in range(6):
        att = closed_loop_pool(rng, T6, K, 40, J)
        ben = benign_humans(rng, 220, T6, offset=40)
        log = concat(att, ben)
        a = log[0]
        seed_ids = np.r_[0:20, 40:60]                         # half the campaign + 20 benign wrongly linked
        cand_ids = np.r_[20:40, 60:260]
        ms = np.isin(a, seed_ids); mc = np.isin(a, cand_ids)
        remap = -np.ones(260, int); remap[cand_ids] = np.arange(len(cand_ids))
        sc = corr_with_pool((a[ms], log[1][ms], log[2][ms]), (remap[a[mc]], log[1][mc], log[2][mc]), T0, T6, tau=60.0, n_cand=len(cand_ids))
        lab = np.r_[np.ones(20), np.zeros(200)]
        sc = np.nan_to_num(-sc, nan=0.0)
        aucs.append(roc_auc_score(lab, sc))
    growth.append(dict(J=J, K=K, auc=round(float(np.mean(aucs)), 3)))
RES["pool_growth_auc"] = growth
print("pool growth:", growth)


In [ ]:
# Cell 7 — summary to paste back
RES["runtime_min"] = round((time.time() - T_START) / 60, 1)
with open("nb1_results.json", "w") as f:
    json.dump(RES, f, indent=1)
print("===== PASTE FROM HERE =====")
print(json.dumps(RES, separators=(",", ":")))
print("===== PASTE TO HERE =====")
